In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

RAW_PATH = os.path.join(PROJECT_PATH, "data/raw")
PROCESSED_PATH = os.path.join(PROJECT_PATH, "data/processed")
EXTERNAL_PATH = os.path.join(PROJECT_PATH, "data/external")
MODEL_PATH = os.path.join(PROJECT_PATH, "models")
FIGURE_PATH = os.path.join(PROJECT_PATH, "reports/figures")
TABLE_PATH = os.path.join(PROJECT_PATH, "reports/tables")
OUTPUT_PATH = os.path.join(PROJECT_PATH, "outputs")

print("Project connected successfully.")
print(PROJECT_PATH)

In [ ]:
import os

print("Raw data files:")

for file in os.listdir(RAW_PATH):
    print("-", file)

In [ ]:
import pandas as pd
import numpy as np

stations = pd.read_csv(
    os.path.join(RAW_PATH, "stations.csv")
)

print("Shape:", stations.shape)

In [ ]:
stations.info()

In [ ]:
display(stations.head())

In [ ]:
station_hour_path = os.path.join(
    RAW_PATH,
    "station_hour.csv"
)

df = pd.read_csv(
    station_hour_path,
    low_memory=False
)

print("Hourly dataset loaded.")
print("Shape:", df.shape)

In [ ]:
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
display(df.head())

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
datetime_check = pd.to_datetime(
    df["Datetime"],
    errors="coerce"
)

print("Earliest:", datetime_check.min())
print("Latest:", datetime_check.max())
print("Invalid timestamps:", datetime_check.isna().sum())

In [ ]:
print("Unique stations in hourly data:", df["StationId"].nunique())
print("Stations in metadata:", stations["StationId"].nunique())

In [ ]:
hourly_ids = set(df["StationId"].dropna().unique())
metadata_ids = set(stations["StationId"].dropna().unique())

missing_metadata = hourly_ids - metadata_ids

print("Stations without metadata:", len(missing_metadata))

if missing_metadata:
    print(sorted(missing_metadata))
else:
    print("All hourly stations have metadata.")

In [ ]:
station_metadata = stations[
    ["StationId", "StationName", "City", "State", "Status"]
].copy()

display(station_metadata.head())

In [ ]:
df_merged = df.merge(
    station_metadata,
    on="StationId",
    how="left",
    validate="many_to_one"
)

print("Original rows:", len(df))
print("Merged rows:  ", len(df_merged))

In [ ]:
print("Missing StationName:", df_merged["StationName"].isna().sum())
print("Missing City:", df_merged["City"].isna().sum())
print("Missing State:", df_merged["State"].isna().sum())

In [ ]:
cities = sorted(
    df_merged["City"].dropna().unique()
)

print("Number of cities:", len(cities))
print(cities)

In [ ]:
delhi_df = df_merged[
    df_merged["City"] == "Delhi"
].copy()

print("Delhi shape:", delhi_df.shape)

In [ ]:
delhi_df["Datetime"] = pd.to_datetime(
    delhi_df["Datetime"],
    errors="coerce"
)

print("Datetime dtype:", delhi_df["Datetime"].dtype)
print("Invalid timestamps:", delhi_df["Datetime"].isna().sum())

In [ ]:
print("Delhi start:", delhi_df["Datetime"].min())
print("Delhi end:  ", delhi_df["Datetime"].max())

In [ ]:
print(
    "Number of Delhi stations:",
    delhi_df["StationId"].nunique()
)

In [ ]:
delhi_stations = (
    delhi_df[
        ["StationId", "StationName"]
    ]
    .drop_duplicates()
    .sort_values("StationId")
    .reset_index(drop=True)
)

display(delhi_stations)

In [ ]:
required_columns = [
    "AQI",
    "AQI_Bucket",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Xylene"
]

for column in required_columns:
    status = "FOUND" if column in delhi_df.columns else "MISSING"
    print(f"{column}: {status}")

In [ ]:
phase1_path = os.path.join(
    PROCESSED_PATH,
    "delhi_merged_raw.parquet"
)

delhi_df.to_parquet(
    phase1_path,
    index=False
)

print("Saved successfully:")
print(phase1_path)

In [ ]:
print("=" * 50)
print("PHASE 1 — DATA UNDERSTANDING COMPLETE")
print("=" * 50)

print("Total raw records:       ", len(df))
print("Total raw stations:      ", df["StationId"].nunique())
print("Total cities:            ", df_merged["City"].nunique())
print("Delhi records:           ", len(delhi_df))
print("Delhi stations:          ", delhi_df["StationId"].nunique())
print("Delhi start:             ", delhi_df["Datetime"].min())
print("Delhi end:               ", delhi_df["Datetime"].max())
print("Delhi dataset saved to:  ", phase1_path)

print("=" * 50)

In [ ]:
import os
import json
import pandas as pd

# Project paths
PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save Delhi dataset
# --------------------------------------------------

delhi_checkpoint = os.path.join(
    PROCESSED_PATH,
    "delhi_phase1_checkpoint.parquet"
)

delhi_df.to_parquet(
    delhi_checkpoint,
    index=False
)

# --------------------------------------------------
# 2. Save station metadata
# --------------------------------------------------

station_checkpoint = os.path.join(
    PROCESSED_PATH,
    "station_metadata_checkpoint.parquet"
)

station_metadata.to_parquet(
    station_checkpoint,
    index=False
)

# --------------------------------------------------
# 3. Save project configuration
# --------------------------------------------------

config = {
    "PROJECT_PATH": PROJECT_PATH,
    "PROCESSED_PATH": PROCESSED_PATH,
    "OUTPUT_PATH": OUTPUT_PATH
}

config_file = os.path.join(
    OUTPUT_PATH,
    "project_config.json"
)

with open(config_file, "w") as f:
    json.dump(config, f, indent=4)

# --------------------------------------------------
# 4. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": "Phase 1",
    "next_phase": "Phase 2 - Data Quality Audit",
    "delhi_records": int(len(delhi_df)),
    "delhi_stations": int(delhi_df["StationId"].nunique()),
    "start_datetime": str(delhi_df["Datetime"].min()),
    "end_datetime": str(delhi_df["Datetime"].max())
}

status_file = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(status_file, "w") as f:
    json.dump(session_status, f, indent=4)

# --------------------------------------------------
# 5. Verify files
# --------------------------------------------------

print("CHECKPOINT SAVED SUCCESSFULLY")
print("-" * 50)

print("Delhi dataset:")
print(delhi_checkpoint)

print("\nStation metadata:")
print(station_checkpoint)

print("\nProject configuration:")
print(config_file)

print("\nSession status:")
print(status_file)

print("-" * 50)
print("Completed: Phase 1")
print("Next: Phase 2 - Data Quality Audit")

In [ ]:
import os

files_to_check = [
    delhi_checkpoint,
    station_checkpoint,
    config_file,
    status_file
]

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )